# **Explore prompt engineering techniques to optimize LLM responses for different applications.**

## Overview
This notebook demonstrates various **prompt engineering techniques** to optimize Large Language Model (LLM) responses for different applications. We'll explore 5 key techniques using Microsoft's Phi-3 model:

### Learning Objectives:
1. **Understand prompt engineering fundamentals** - How different prompting strategies affect model output
2. **Implement role assignment** - Guide the model to adopt specific personas or expertise levels
3. **Apply chain-of-thought prompting** - Break down complex reasoning into step-by-step processes
4. **Use output constraints** - Control response format, length, and structure
5. **Leverage few-shot learning** - Provide examples to guide model behavior

### Techniques Covered:
- **Role Assignment** - Making the AI adopt specific personas
- **Chain-of-Thought** - Step-by-step reasoning
- **Output Constraints** - Controlling response format
- **Hypothetical Scenarios** - Context-based prompting
- **Few-Shot Learning** - Learning from examples

---

## Environment Setup

Before we begin exploring prompt engineering techniques, we need to install the required dependencies:

### Required Libraries:
- **torch** - PyTorch deep learning framework for tensor operations
- **transformers** - Hugging Face library for pre-trained language models
- **accelerate** - Library for distributed training and inference optimization
- **bitsandbytes** - Enables efficient 4-bit quantization to reduce memory usage

### Why These Dependencies?
1. **Memory Efficiency**: 4-bit quantization allows us to run large models on consumer hardware
2. **Model Access**: Transformers library provides easy access to state-of-the-art models
3. **Performance**: Accelerate optimizes inference speed and memory usage

In [ ]:
# Install core dependencies for LLM inference
# torch: PyTorch framework for deep learning operations
# transformers: Hugging Face library for accessing pre-trained models
# accelerate: Optimizes model loading and inference across devices
# bitsandbytes: Enables 4-bit quantization for memory-efficient inference
!pip install torch transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.0 MB/s eta 0:00:00


In [ ]:
# Update bitsandbytes to the latest version for optimal 4-bit quantization support
# The -U flag ensures we get the most recent version with latest optimizations
!pip install -U bitsandbytes

## Model Setup and Prompt Engineering Implementation

### Model Choice: Microsoft Phi-3-mini-4k-instruct
We're using **Phi-3-mini-4k-instruct** because:
- **Compact Size**: ~3.8B parameters, suitable for consumer hardware
- **Instruction-Tuned**: Optimized for following complex prompts and instructions
- **4K Context**: Can handle moderately long conversations and contexts
- **High Quality**: Competitive performance despite smaller size

### Key Implementation Features:

#### **Model Configuration**
- **4-bit Quantization**: Reduces memory usage by ~75% while maintaining quality
- **Float16 Precision**: Balances speed and accuracy
- **Auto Device Mapping**: Automatically distributes model across available GPUs/CPU

#### **Inference Function**
- **Configurable Generation**: Temperature control for creativity vs consistency
- **Token Limits**: Prevents runaway generation with max_new_tokens
- **Clean Output**: Removes special tokens for readable responses

#### **Prompt Engineering Techniques**
Each technique demonstrates a different approach to guide model behavior:

1. **Role Assignment** - Assigns expertise and target audience
2. **Chain-of-Thought** - Encourages step-by-step reasoning
3. **Output Constraints** - Controls format and length precisely
4. **Hypothetical Scenarios** - Provides creative context
5. **Few-Shot Learning** - Shows examples to guide behavior

In [ ]:
# Cell 1: Model Loading and Setup with fix for RoPE scaling error
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig
import torch

if not torch.cuda.is_available():
    print("ERROR: Please enable GPU (Runtime -> Change runtime type -> T4 GPU).")
else:
    model_name = "microsoft/Phi-3-mini-4k-instruct"

    # Define the quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    print(f"Loading {model_name} configuration...")

    # Fix for ValueError: Unknown RoPE scaling type default
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

    # In some versions, transformers sets rope_scaling to {'type': 'default'}
    # Phi-3's remote code does not recognize 'default' and crashes.
    if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
        if config.rope_scaling.get("type") == "default" or "rope_type" in config.rope_scaling:
            config.rope_scaling = None
            print("Adjusted RoPE scaling to prevent initialization error.")

    print(f"Loading {model_name} with 4-bit quantization...")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        config=config,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,
        attn_implementation="eager"
    )
    print("Model loaded successfully!")

Loading microsoft/Phi-3-mini-4k-instruct configuration...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Adjusted RoPE scaling to prevent initialization error.
Loading microsoft/Phi-3-mini-4k-instruct with 4-bit quantization...


... (10 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
# Cell 2: Run Prompt Engineering Experiments
def run_inference(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

techniques = {
    "1. Role Assignment": "You are a professional science teacher. Explain quantum computing to a 10-year-old in 3 short bullet points.",
    "2. Step-by-Step": "Explain quantum computing step-by-step, then provide a one-sentence summary.",
    "3. Constraints": "Explain quantum computing in exactly 3 sentences.",
    "4. Scenario": "Explain quantum computing as a sports commentator to football fans.",
    "5. Few-Shot": "Example: Classical is off/on. Example: Quantum is a dimmer. Give 2 more analogies."
}

for name, prompt in techniques.items():
    print(f"\n=== {name} ===")
    print(run_inference(prompt))
    print("-" * 40)


=== 1. Role Assignment ===


AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'